In [1]:
from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q tiktoken numpy

In [ ]:
# Too much output. need to process in batches.
# import os
# import numpy as np
# import tiktoken
# from datasets import load_dataset

# from huggingface_hub import login
# login(token=os.getenv("HF_API_KEY"))

# # Load TinyStories from HuggingFace
# print("Downloading TinyStories...")
# dataset = load_dataset("roneneldan/TinyStories", split="train")
# val_dataset = load_dataset("roneneldan/TinyStories", split="validation")
# print(f"Train stories: {len(dataset):,}")
# print(f"Val stories: {len(val_dataset):,}")

# # Use GPT-2 tokenizer (same as nanoGPT)
# enc = tiktoken.get_encoding("gpt2")

# def tokenize(examples):
#     # Join all stories with EOS token between them
#     ids = []
#     for text in examples["text"]:
#         ids.extend(enc.encode_ordinary(text))
#         ids.append(enc.eot_token)  # end of text token between stories
#     return ids

# # Tokenize
# print("Tokenizing train split...")
# train_ids = tokenize(dataset)
# print("Tokenizing val split...")
# val_ids = tokenize(val_dataset)

# print(f"Train tokens: {len(train_ids):,}")
# print(f"Val tokens: {len(val_ids):,}")

# # Save as binary files (same format nanoGPT expects)
# data_dir = '/content/drive/MyDrive/nanoGPT/data/tinystories'
# os.makedirs(data_dir, exist_ok=True)

# train_arr = np.array(train_ids, dtype=np.uint16)
# val_arr = np.array(val_ids, dtype=np.uint16)

# train_arr.tofile(os.path.join(data_dir, 'train.bin'))
# val_arr.tofile(os.path.join(data_dir, 'val.bin'))

# print(f"Saved to {data_dir}")
# print(f"Train: {train_arr.nbytes / 1e6:.1f} MB")
# print(f"Val: {val_arr.nbytes / 1e6:.1f} MB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

...

Generating validation split: 0 examples [00:00, ? examples/s]

Train stories: 2,119,719
Val stories: 21,990
Tokenizing train split...


: 

: 

: 

In [1]:
import os
import numpy as np
import tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
data_dir = '/content/drive/MyDrive/nanoGPT/data/tinystories'
os.makedirs(data_dir, exist_ok=True)

from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

def tokenize_and_save(split, output_path):
    print(f"Processing {split}...")
    dataset = load_dataset("roneneldan/TinyStories", split=split)
    
    # Write in chunks directly to disk instead of holding everything in memory
    chunk_size = 50000
    all_ids = []
    
    for i, example in enumerate(dataset):
        ids = enc.encode_ordinary(example["text"])
        ids.append(enc.eot_token)
        all_ids.extend(ids)
        
        # Flush to disk every chunk_size stories
        if (i + 1) % chunk_size == 0:
            arr = np.array(all_ids, dtype=np.uint16)
            with open(output_path, 'ab') as f:
                arr.tofile(f)
            all_ids = []
            print(f"  {i+1:,} stories processed...")
    
    # Write remaining
    if all_ids:
        arr = np.array(all_ids, dtype=np.uint16)
        with open(output_path, 'ab') as f:
            arr.tofile(f)
    
    print(f"Saved to {output_path}")

# Remove any partial files from the crashed run
for f in ['train.bin', 'val.bin']:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        os.remove(path)

tokenize_and_save("train", os.path.join(data_dir, 'train.bin'))
tokenize_and_save("validation", os.path.join(data_dir, 'val.bin'))

# Check file sizes
for f in ['train.bin', 'val.bin']:
    path = os.path.join(data_dir, f)
    print(f"{f}: {os.path.getsize(path) / 1e6:.1f} MB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Processing train...
  50,000 stories processed...
  100,000 stories processed...
  150,000 stories processed...
  200,000 stories processed...
  250,000 stories processed...
  300,000 stories processed...
  350,000 stories processed...
  400,000 stories processed...
  450,000 stories processed...
  500,000 stories processed...
  550,000 stories processed...
  600,000 stories processed...
  650,000 stories processed...
  700,000 stories processed...
  750,000 stories processed...
  800,000 stories processed...
  850,000 stories processed...
  900,000 stories processed...
  950,000 stories processed...
  1,000,000 stories processed...
  1,050,000 stories processed...
  1,100,000 stories processed...
  1,150,000 stories processed...
  1,200,000 stories processed...
  1,250,000 stories processed...
  1,300,000 stories processed...
  1,350,000 stories processed...
  1,400,000 stories processed...
  1,450,000 stories processed...
  1,500,000 stories processed...
  1,550,000 stories processed

In [5]:
from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 7 (delta 1), reused 7 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 10.48 KiB | 104.00 KiB/s, done.
From https://github.com/derekn4/llm-training-pipeline
   2b98fc4..5b64f91  main       -> origin/main
Updating 2b98fc4..5b64f91
Fast-forward
 .gitignore        |   6 +-
 00_setup.ipynb    |  46 +++++-
 01_pretrain.ipynb | 482 +++++++++++++++++++++++++++++++++++++++++++++++++++++-
 configurator.py   |  47 ++++++
 model.py          | 330 +++++++++++++++++++++++++++++++++++++
 5 files changed, 906 insertions(+), 5 deletions(-)
 create mode 100644 configurator.py
 create mode 100644 model.py


In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import math
import time
import os
import wandb

from model import GPTConfig, GPT

In [7]:
wandb.login(key=os.getenv("WANDB_API_KEY"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: derek-nguyen99 (derek-nguyen99-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
# Model architecture
n_layer = 6         #number of layers in neural network. each layer is 1 full transformer block
n_head = 6          #number of attention heads in each transformer block
n_embd = 384        #number of embedding dimensions 
block_size = 256    #maximum context length for predictions
dropout = 0.0       #dropout rate for regularization
bias = False        #whether to use bias in the linear layers
vocab_size = 50304  # GPT-2 vocab rounded up to nearest multiple of 64

# Data
data_dir = 'data/tinystories'
batch_size = 32                     #number of sequences per batch
gradient_accumulation_steps = 8     # effective batch = 32 * 8 * 256 = ~65k tokens

# Training
max_iters = 5000        #maximum number of training iterations
eval_interval = 500     #evaluation interval
eval_iters = 50         #number of iterations to run during evaluation
log_interval = 10       #logging interval

# Optimizer
learning_rate = 3e-4    #initial learning rate
weight_decay = 1e-1     #weight decay for regularization
beta1 = 0.9             #exponential decay rate for the first moment estimates in Adam optimizer
beta2 = 0.95            #exponential decay rate for the second moment estimates in Adam optimizer
grad_clip = 1.0         #maximum gradient norm for gradient clipping

# LR schedule
warmup_iters = 200
min_lr = 3e-5

# System
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16

print(f"Device: {device}")
print(f"Effective batch size: {batch_size * gradient_accumulation_steps * block_size:,} tokens")

Device: cuda
Effective batch size: 65,536 tokens


In [ ]:
# Get a batch of data

def get_batch(split):
    path = os.path.join(data_dir, f'{split}.bin')
    data = np.memmap(path, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y

# Test it
x, y = get_batch('train')
print(f"x shape: {x.shape}")  # should be [32, 256]
print(f"y shape: {y.shape}")  # should be [32, 256]
print(f"x dtype: {x.dtype}")
print(f"Sample tokens: {x[0, :10]}")  # first 10 tokens of first sequence

x shape: torch.Size([32, 256])
y shape: torch.Size([32, 256])
x dtype: torch.int64
Sample tokens: tensor([  329,   465,  1037,   290, 41130,   683, 17707,    13,  3574,   326],
       device='cuda:0')


In [ ]:
# Test encoding and decoding
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.decode(x[0, :10].tolist()))

 for his help and hugged him tightly. From that


In [11]:
# Initialize model
model_args = dict(
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    block_size=block_size,
    bias=bias,
    vocab_size=vocab_size,
    dropout=dropout,
)

gptconf = GPTConfig(**model_args)
model = GPT(gptconf)
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print(f"Model device: {next(model.parameters()).device}")

number of parameters: 29.94M
Model parameters: 30.04M
Model device: cuda:0


In [ ]:
# Initialize optimizer
optimizer = model.configure_optimizers(
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    betas=(beta1, beta2),
    device_type='cuda'
)

print("Optimizer initialized ✓")

num decayed parameter tensors: 26, with 30,031,872 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True
Optimizer initialized ✓


In [13]:
# Learning rate schedule with linear warmup and cosine decay
def get_lr(it):
    # Linear warmup
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    # Minimum LR after decay
    if it > max_iters:
        return min_lr
    # Cosine decay in between
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# Sanity check -- plot a few values
print(f"LR at iter 0:    {get_lr(0):.6f}")
print(f"LR at iter 100:  {get_lr(100):.6f}")
print(f"LR at iter 200:  {get_lr(200):.6f}  ← end of warmup")
print(f"LR at iter 2500: {get_lr(2500):.6f} ← halfway")
print(f"LR at iter 5000: {get_lr(5000):.6f} ← end")

LR at iter 0:    0.000001
LR at iter 100:  0.000151
LR at iter 200:  0.000300  ← end of warmup
LR at iter 2500: 0.000174 ← halfway
LR at iter 5000: 0.000030 ← end


In [15]:
# Mixed precision context
ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16)
scaler = torch.amp.GradScaler('cuda')

# Evaluation function
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Eval function ready ✓")

Eval function ready ✓


In [ ]:
# Initialize wandb
wandb.init(
    project="llm-training-pipeline",
    name="tinystories-10M-baseline",
    config={
        "n_layer": n_layer,
        "n_head": n_head,
        "n_embd": n_embd,
        "block_size": block_size,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "max_iters": max_iters,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "warmup_iters": warmup_iters,
        "dataset": "tinystories",
        "n_params": n_params,
    }
)

# Training state
iter_num = 0
best_val_loss = float('inf')
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Fetch first batch
X, Y = get_batch('train')
t0 = time.time()

print("Starting training...")
'''
Training Loop:
1. set LR
2. maybe evaluate + checkpoint
3. 8x (forward → backward → accumulate gradients → prefetch next batch)
4. unscale gradients
5. clip gradients
6. update weights
7. update scaler
8. zero gradients
9. log
10. increment iter_num
'''

while iter_num <= max_iters:

    # Set LR for this iteration
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Evaluate and checkpoint every eval_interval steps
    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.6f}")
        wandb.log({
            "iter": iter_num,
            "train/loss": losses['train'],
            "val/loss": losses['val'],
            "lr": lr,
        })

        # Save checkpoint if best val loss
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            checkpoint = {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'model_args': model_args,
                'iter_num': iter_num,
                'best_val_loss': best_val_loss,
            }
            ckpt_path = os.path.join(checkpoint_dir, 'ckpt_best.pt')
            torch.save(checkpoint, ckpt_path)
            print(f"  → checkpoint saved (val loss {best_val_loss:.4f})")

    # Forward + backward with gradient accumulation
    for micro_step in range(gradient_accumulation_steps):
        with ctx:
            logits, loss = model(X, Y)
            loss = loss / gradient_accumulation_steps
        X, Y = get_batch('train')  # prefetch next batch
        scaler.scale(loss).backward()

    # Gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    # Optimizer step
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # Logging
    t1 = time.time()
    dt = t1 - t0
    t0 = t1

    if iter_num % log_interval == 0:
        lossf = loss.item() * gradient_accumulation_steps
        print(f"iter {iter_num}: loss {lossf:.4f}, time {dt*1000:.2f}ms, lr {lr:.6f}")
        wandb.log({
            "iter": iter_num,
            "train/loss_step": lossf,
            "lr": lr,
        })

    iter_num += 1

print("Training complete!")
wandb.finish()